# Exercise 7.2: A simple voltage clamp model

In this exercise, we will explore how the cell behaves as a capacitor. We look at the cell as only the impermeable membrane, with no ion channels present. We then look at an experimental setup where we insert a small pipette through the membrane and into the cell. With this probe, we can measure the membrane potential using a voltmeter.

We will look at how this pipette can be used to implement a **voltage clamp**. A voltage clamp is a highly useful experimental setup used to artificially control the membrane potential. If our voltmeter measures a membrane potential different from our desired target potential, a bit of applied current is injected through the pipette, changing the membrane potential as a result.

We can think of the situation as the following electric circuit:

![Voltage clamp circuit](../../fig/voltage_clamp_circuit.png)

## The differential equations

We know that the electric potential across the two components must be the same, and this is the membrane potential $V$. We want to find this potential as a function of time, $V(t)$. Ideally, it should follow our prescribed target, $V_{\mathrm{target}}(t)$, which can be constant in time or any time-dependent function we want.

Our pipette acts as an Ohmic current source with a magnitude given by:

$$
I_{\mathrm{applied}} = \frac{V - V_{\mathrm{target}}}{R_{\mathrm{s}}}
$$

Note that this current is larger if the membrane potential is far from its target, and it dies out once the potential approaches its target value.

Any current across the pipette will lead to a similar capacitive current across the membrane, which will change the charge built up across the membrane. So we have:

$$
C_{\mathrm{m}} \frac{\mathrm{d}V}{\mathrm{d}t} = I_{\mathrm{cap}} = -I_{\mathrm{applied}}
$$

Giving an ODE for the membrane potential:

$$
\frac{\mathrm{d}V}{\mathrm{d}t} = \frac{V_{\mathrm{target}}(t) - V}{C_{\mathrm{m}} R_{\mathrm{s}}}
$$

If you are not used to analyzing electrical circuits and found this hard to follow, do not worry! The most important takeaway is that we found an ODE describing how the membrane potential changes over time, which you will now solve.


## Exercise 7.2a: Implementing the target potential

Let the prescribed membrane potential be the following step function:

$$
V_{\mathrm{target}}(t) =
\begin{cases}
-40 \text{ mV} & \text{if } 2 \text{ ms} < t < 6 \text{ ms}, \\
-80 \text{ mV} & \text{otherwise.}
\end{cases}
$$

Implement this as a Python function `V_target(t)` and plot it for the period $t \in [0, 10]$ ms.

_Hint: Rather than use `if`-tests, you can use boolean math: `x _ (t > 2) _ (t < 6)`. This allows your function to take in both a scalar `t` and a full NumPy time array._


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def V_target(t):
    # Your code here
    return ...


t = np.linspace(...)

plt.plot(t, V_target(t))
plt.show()

## Exercise 7.2b: Checking the units

Let $V$ be denoted in millivolts (mV) and time in milliseconds (ms). Let $C_{\mathrm{m}} = 0.05$ nF and $R_{\mathrm{s}} = 10$ MΩ.

Look at the ODE and make sure the units are consistent by hand. Make any necessary changes to the units before writing your code.

$$
\frac{\mathrm{d}V}{\mathrm{d}t} = \frac{V_{\mathrm{target}} - V}{C_{\mathrm{m}} R_{\mathrm{s}}}
$$


In [ ]:
# Write your unit conversions/checks here if needed

## Exercise 7.2c: Solving the ODE numerically

Now, implement the right-hand side (RHS) of the ODE system and use `scipy.integrate.solve_ivp` to solve it to find the membrane potential $V(t)$ in the period $t \in [0, 10]$ ms.

Let the parameters and units be as given in (7.2b) and the initial condition be $V_0 = -80$ mV. Plot your solution.


In [ ]:
from scipy.integrate import solve_ivp


def rhs(t, V, Cm, Rs):
    # Your code here
    dV_dt = ...
    return dV_dt


In [ ]:
# Define time array
time_span = ...

# Define parameters
Cm = ...
Rs = ...
params = (Cm, Rs)

# Define initial condition
y0 = ...

# Call solve_ivp
solution = solve_ivp(..., ..., ..., args=params, max_step=...)

In [ ]:
# Plot solution
# ...

## Exercise 7.2d: Analyzing the solution

Compare your computed solution for $V(t)$ to your plot of the prescribed $V_{\mathrm{target}}(t)$. Describe the differences.

_(To make the comparison easier, it might be a nice idea to plot the two curves on top of each other in the same figure)._


In [ ]:
# Your analysis/plotting here

## (Optional) Exercise 7.2e: Finding the analytical solution

Solve the ODE by hand. What kind of ODE/solution is this? What is the most important functional parameter in this solution?

_Hint: The equation is an example of a **separable** differential equation._


## Exercise 7.2f: Playing around with the model parameters

Run the interactive widget below, and see how the membrane potential response changes as you modify the membrane capacitance $C_{\mathrm{m}}$ and the series resistance $R_{\mathrm{s}}$. _(You might need to run the cell twice for the widget to appear)._

If you want the membrane potential to follow the prescribed voltage as closely as possible, what restrictions does this put on $C_{\mathrm{m}}$ and $R_{\mathrm{s}}$?


In [1]:
import L3_widgets

L3_widgets.VoltageClampWidget().display()

interactive(children=(FloatSlider(value=0.05, description='Cm', max=0.1, min=0.005, step=0.005), IntSlider(val…

## Discussion on Exercise 7.2

As you have probably seen by playing with the widget, for the voltage to accurately follow the prescribed voltage, we want both the membrane capacitance and the series resistance to be as small as possible. In fact, it is the _product_ of the two that is most important, as this defines the characteristic time constant of the membrane response:

$$
\tau = C_{\mathrm{m}} \cdot R_{\mathrm{s}}
$$

If you solved (7.2e), you might have noticed this already. Simply put, the model mathematically dictates that the actual membrane potential will approach the prescribed potential as an exponential relaxation, with a relaxation time of $\tau$. If $\tau$ is large, we will need to wait a long time for it to reach the targeted value.

For a whole-cell voltage clamp protocol, the cell membrane capacitance is typically 0.05 nF, and the series resistance is usually in the range of 5–20 MΩ. This gives a noticeable delay in the membrane potential response, alongside capacitive currents that persist over several milliseconds. This delay can actually disturb the measurement of other ionic currents, introducing unwanted noise.

In another technique—the 'patch' clamp—a microscopic patch of the membrane is torn off (~1 µm²). This means the capacitance $C_{\mathrm{m}}$ is drastically lowered (~0.01 pF). In a patch clamp, the series resistance is usually in the GΩ range, so the time constant becomes very small, resulting in almost zero noise from capacitive currents!
